# SELV CLV MODEL - COMPLETE TEST NOTEBOOK

In [1]:
import pandas as pd
import numpy as np
import joblib
import json

In [2]:
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              mean_absolute_error, mean_squared_error,
                              r2_score)

In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [4]:
from sklearn.preprocessing import MinMaxScaler

In [5]:
# ── Step 1: Load saved models ────────────────────────────────
churn_model = joblib.load('churn_model.pkl')
selv_model  = joblib.load('selv_model.pkl')


In [6]:
# ── Step 2: Load master DataFrame ───────────────────────────
master = pd.read_csv('master_for_testing.csv')


In [7]:
# ── Step 3: Load features ────────────────────────────────────
with open('features.json', 'r') as f:
    features = json.load(f)


In [8]:
# ── Step 4: Recreate X, y from master ───────────────────────
X       = master[features]
y_churn = master['is_churned']
y_selv  = master['SELV_score']


In [9]:
# ── Step 5: Recreate train/test split ────────────────────────
X_train_idx = np.load('X_train_idx.npy', allow_pickle=True)
X_test_idx  = np.load('X_test_idx.npy',  allow_pickle=True)

X_train = X.loc[X_train_idx]
X_test  = X.loc[X_test_idx]
yc_tr   = y_churn.loc[X_train_idx]
yc_te   = y_churn.loc[X_test_idx]
ys_tr   = y_selv.loc[X_train_idx]
ys_te   = y_selv.loc[X_test_idx]


In [10]:
# ── TEST 1: CHURN CLASSIFIER ─────────────────────────────────

In [11]:
print("\n" + "="*50)
print("TEST 1 — CHURN CLASSIFIER")
print("="*50)

yc_pred = churn_model.predict(X_test)
yc_prob = churn_model.predict_proba(X_test)[:,1]
print(f"Accuracy : {accuracy_score(yc_te, yc_pred)*100:.2f}%")
print(f"AUC-ROC  : {roc_auc_score(yc_te, yc_prob):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(yc_te, yc_pred))
print(f"\nClassification Report:")
print(classification_report(yc_te, yc_pred,
      target_names=['Retained', 'Churned']))


TEST 1 — CHURN CLASSIFIER
Accuracy : 99.88%
AUC-ROC  : 1.0000

Confusion Matrix:
[[584   0]
 [  1 273]]

Classification Report:
              precision    recall  f1-score   support

    Retained       1.00      1.00      1.00       584
     Churned       1.00      1.00      1.00       274

    accuracy                           1.00       858
   macro avg       1.00      1.00      1.00       858
weighted avg       1.00      1.00      1.00       858



In [12]:
# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


cv_auc = cross_val_score(churn_model, X, y_churn,
                          cv=cv, scoring='roc_auc')
print(f"CV AUC : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"Stable : {cv_auc.std() < 0.05}")

CV AUC : 1.0000 ± 0.0000
Stable : True


In [13]:
# ── TEST 2: SELV REGRESSOR ───────────────────────────────────

In [14]:
print("\n" + "="*50)
print("TEST 2 — SELV REGRESSOR")
print("="*50)


TEST 2 — SELV REGRESSOR


In [15]:
ys_pred = selv_model.predict(X_test)

mae  = mean_absolute_error(ys_te, ys_pred)
rmse = mean_squared_error(ys_te, ys_pred)**0.5
r2   = r2_score(ys_te, ys_pred)
mape = np.mean(np.abs((ys_te - ys_pred) / (ys_te + 1e-9))) * 100

print(f"R²   : {r2:.4f}  {'✓ Good' if r2>=0.80 else '⚠ Needs improvement'}")
print(f"MAE  : {mae:.4f} SELV score points")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%  {'✓ Good' if mape<20 else '⚠ High'}")

cv_r2 = cross_val_score(selv_model, X, y_selv,cv=5, scoring='r2')
print(f"\nCV R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"Stable: {cv_r2.std() < 0.05}")


R²   : 0.4880  ⚠ Needs improvement
MAE  : 0.1853 SELV score points
RMSE : 1.6217
MAPE : 1.95%  ✓ Good

CV R²: 0.7178 ± 0.2100
Stable: False


In [16]:
# ── TEST 3: FEATURE IMPORTANCE ───────────────────────────────

In [17]:
print("\n" + "="*50)
print("TEST 3 — FEATURE IMPORTANCE")
print("="*50)

fi_churn = pd.Series(churn_model.feature_importances_,
                     index=features).sort_values(ascending=False)
fi_selv  = pd.Series(selv_model.feature_importances_,
                     index=features).sort_values(ascending=False)
print("\nChurn model top features:")
for feat, imp in fi_churn.items():
    bar = '█' * int(imp * 40)
    print(f"  {feat:30s}: {imp:.4f} {bar}")

print("\nSELV model top features:")
for feat, imp in fi_selv.items():
    bar = '█' * int(imp * 40)
    print(f"  {feat:30s}: {imp:.4f} {bar}")


TEST 3 — FEATURE IMPORTANCE

Churn model top features:
  Recency                       : 0.9850 ███████████████████████████████████████
  Monetary                      : 0.0048 
  social_impact                 : 0.0022 
  carbon_emission_kg            : 0.0021 
  environmental_impact          : 0.0018 
  Frequency                     : 0.0018 
  financial_value               : 0.0014 
  Carbon_Reduction_Ratio        : 0.0009 
  RFM_score                     : 0.0000 

SELV model top features:
  social_impact                 : 0.2861 ███████████
  Carbon_Reduction_Ratio        : 0.2496 █████████
  financial_value               : 0.1959 ███████
  environmental_impact          : 0.1113 ████
  Monetary                      : 0.0533 ██
  Recency                       : 0.0515 ██
  Frequency                     : 0.0401 █
  carbon_emission_kg            : 0.0108 
  RFM_score                     : 0.0014 


In [18]:
# ── TEST 4: BUSINESS LOGIC ───────────────────────────────────
print("\n" + "="*50)
print("TEST 4 — BUSINESS LOGIC VALIDATION")
print("="*50)

# Weight constraint
alpha, beta, gamma = 0.35, 0.35, 0.30
print(f"Weight constraint : α+β+γ = {alpha+beta+gamma} ✓")

# SELV range check
print(f"SELV range        : {master['SELV_score'].min():.2f} — {master['SELV_score'].max():.2f}")
print(f"Expected range    : 0 to 100")
print(f"Range OK          : {master['SELV_score'].max() <= 100}")

# Segment distribution
print(f"\nSegment distribution:")
print(master['selv_segment'].value_counts())

# Priority matrix
q75_selv  = master['SELV_score'].quantile(0.75)
churn_all = churn_model.predict_proba(X)[:,1]
selv_all  = selv_model.predict(X)

critical = ((churn_all > 0.7) & (selv_all > q75_selv)).sum()
loyal    = ((churn_all < 0.3) & (selv_all > q75_selv)).sum()
let_go   = ((churn_all > 0.7) & (selv_all < master['SELV_score'].quantile(0.25))).sum()
nurture  = ((churn_all < 0.3) & (selv_all < master['SELV_score'].quantile(0.25))).sum()

print(f"\nPriority Matrix:")
print(f"  CRITICAL (urgent retention) : {critical}")
print(f"  LOYAL    (keep happy)       : {loyal}")
print(f"  LET GO   (low priority)     : {let_go}")
print(f"  NURTURE  (upsell)           : {nurture}")


TEST 4 — BUSINESS LOGIC VALIDATION
Weight constraint : α+β+γ = 1.0 ✓
SELV range        : 0.37 — 44.44
Expected range    : 0 to 100
Range OK          : True

Segment distribution:
selv_segment
Needs Nurturing            2876
At Risk                    1410
High Value High Emitter       3
Name: count, dtype: int64

Priority Matrix:
  CRITICAL (urgent retention) : 312
  LOYAL    (keep happy)       : 749
  LET GO   (low priority)     : 340
  NURTURE  (upsell)           : 730


In [19]:
# ── TEST 5: FINAL SUMMARY ────────────────────────────────────
print("\n" + "="*50)
print("FINAL TEST SUMMARY")
print("="*50)

churn_auc = roc_auc_score(yc_te, yc_prob)
selv_r2   = r2_score(ys_te, ys_pred)

tests = {
    'Churn AUC >= 0.80'     : churn_auc >= 0.80,
    'SELV R² >= 0.80'       : selv_r2   >= 0.80,
    'CV churn stable'       : cv_auc.std() < 0.05,
    'CV SELV stable'        : cv_r2.std()  < 0.05,
    'Weight constraint = 1' : round(alpha+beta+gamma, 10) == 1.0,
    'SELV range 0-100'      : master['SELV_score'].max() <= 100,
    'No null features'      : X.isnull().sum().sum() == 0,
}

for test_name, passed in tests.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f"  {test_name:30s}: {status}")

total_pass = sum(tests.values())
print(f"\n{total_pass}/{len(tests)} tests passed")
print("Model ready for production" if total_pass == len(tests)
      else "Fix failing tests before deployment")





FINAL TEST SUMMARY
  Churn AUC >= 0.80             : ✓ PASS
  SELV R² >= 0.80               : ✗ FAIL
  CV churn stable               : ✓ PASS
  CV SELV stable                : ✗ FAIL
  Weight constraint = 1         : ✓ PASS
  SELV range 0-100              : ✓ PASS
  No null features              : ✓ PASS

5/7 tests passed
Fix failing tests before deployment


In [20]:
# #importing libraries
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import MinMaxScaler
# from xgboost import XGBClassifier, XGBRegressor
# # XGBClassifier for decision tree
# from sklearn.model_selection import train_test_split

# import numpy as np


# #load dataset 
# df = pd.read_csv('Online Retail.csv',  encoding='ISO-8859-1')  #most common fix for this dataset



# df.info()

# df['Description'].unique()

# #checking null values
# df.isnull().sum()

# #delete those rows where Customer_ID is null
# df = df.dropna(subset=['CustomerID'])



# #verify the deleted rows
# df.isnull().sum()

# # #Convert InvoiceDate → datetime
# # df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# #Convert InvoiceDate → datetime 
# df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# #total price 
# df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Financial Value

# df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%d-%m-%Y %H:%M')
# snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# #Calculate RFM 
# rfm = df.groupby('CustomerID').agg(
#     Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
#     Frequency = ('InvoiceNo',   'nunique'),
#     Monetary  = ('TotalPrice',     'sum')
# ).reset_index()

# # Financial Value = normalised CLV proxy
# rfm['avg_order_val']  = rfm['Monetary'] / rfm['Frequency']
# rfm['lifespan_days']  = df.groupby('CustomerID')['InvoiceDate'].apply(
#     lambda x: (x.max() - x.min()).days).values

# #calculate clv
# rfm['clv_raw'] = rfm['avg_order_val'] * rfm['Frequency'] * (rfm['lifespan_days'] / 365 + 0.01)

# # Normalise to 0-1
# scaler = MinMaxScaler()
# rfm['financial_value'] = scaler.fit_transform(rfm[['clv_raw']])

# rfm

# CALCULATE ENVIRONMENTAL IMPACAT         
# CarbonReduction = BaselineCarbon - CustomerCarbonFootprint              
# Environmental_Impact = Carbon_Redu
# ction + Green_Purchase_Score                    
# Higher value means more environmentally friendly customer behavior.                  

# #load carbon_factor file
# carbon_factor = pd.read_excel('carbon_factors_output.xlsx')

# carbon_factor

# #merging both files in merge_df dataset
# merged_df = df.merge(
#     carbon_factor[['Description', 'carbon_factor_kg_co2e_per_kg', 'material_matched']],
#     on='Description',
#     how='left'  # keep all retail rows
# )

# #Calculate Carbon Emission = qty * carbon_factor
# merged_df['carbon_emission_kg'] = (
#     merged_df['Quantity'] * merged_df['carbon_factor_kg_co2e_per_kg']
# )


# # Get the indices of the "mixed/unknown" rows and drop them
# merged_df.drop(merged_df[merged_df['material_matched'] == 'DEFAULT (mixed/unknown)'].index, inplace=True)
# merged_df['material_matched'].isnull().sum()

# # merged_df.isnull().sum()

# # Drop rows with any empty cells
# merged_df.dropna(inplace=True)

# # check/verify null cells
# print(merged_df.isnull().sum())


# ## 1.50 is our default mixed/unknown carbon factor
# merged_df["carbon_factor_kg_co2e_per_kg"] = merged_df["carbon_factor_kg_co2e_per_kg"].fillna(0.15)

# #drop the -ve values contained 
# merged_df.drop(merged_df[merged_df["carbon_factor_kg_co2e_per_kg"] < 0].index, inplace=True)
# #delete null value
# merged_df.dropna(inplace=True)


# #for baseline
# #load dataset contains baseline_per_product value
# baseline = pd.read_csv("clean_dataII_with_wri_baseline.csv",  encoding='ISO-8859-1')  #most common fix for this dataset

# baseline


# #create lookup dictionaries
# # baseline_product_map = baseline.set_index("Description")["baseline_product"].to_dict()
# wri_material_map = baseline.set_index("Description")["wri_material_baseline"].to_dict()


# #map columns to merged_df
# # merged_df["baseline_product"] = merged_df["Description"].map(baseline_product_map)

# merged_df["wri_material_baseline"] = merged_df["Description"].map(wri_material_map)

# #handle missing values
# # merged_df["baseline_product"].fillna("other", inplace=True)
# merged_df["wri_material_baseline"] = merged_df["wri_material_baseline"].fillna(0.15)



# #calculate total baseline_carbon_per_totalproduct
# merged_df["baseline_carbon"] = (
#     merged_df["Quantity"] * merged_df["wri_material_baseline"]
# )

# merged_df

# merged_df['Carbon_Reduction'] = merged_df['baseline_carbon'] - merged_df['carbon_emission_kg']
# # Positive Carbon_Reduction = better than baseline (reduced emissions)
# # Negative Carbon_Reduction = worse than baseline (increased emissions)

# merged_df['Carbon_Reduction_Ratio'] = merged_df['Carbon_Reduction']/merged_df['baseline_carbon']

# merged_df['Carbon_Reduction_Ratio'].isna().sum()

# merged_df['environmental_impact'] = scaler.fit_transform(merged_df[['Carbon_Reduction_Ratio']])



# #------------------------------------------------------
# # store necessary columns related to environmentalimapct into env dataset
# env = merged_df.groupby('CustomerID').agg(
#     Carbon_Reduction = ('Carbon_Reduction', 'sum'),
#     carbon_emission_kg = ('carbon_emission_kg', 'sum'),
#     Carbon_Reduction_Ratio = ('Carbon_Reduction_Ratio', 'sum'),
#     environmental_impact = ('environmental_impact', 'sum')
# ).reset_index()

# #----------------------------------------------------------

# # Normalise to 0-1
# env['environment_impact'] = scaler.fit_transform(
#     env[['Carbon_Reduction']]
# )

# env

# SOCIAL IMPACT

# # Normalise and apply Entropy Weights:
# # Features going into social score
# social_features = ['country_diversity',
#                    'engagement_rate',
#                    'product_variety',
#                    'total_quantity']

# # Handle any nulls
# social[social_features] = social[social_features].fillna(0)
# # Normalise to 0-1
# social_norm = social[social_features].copy()
# social_norm[social_features] = scaler.fit_transform(social[social_features])

# # PILLAR 1: Geographic Equity & Inclusion
# # Source: OECD Regional Well-being Framework (2020)
# # "Geographic diversity of customer base reflects
# #  inclusivity and equitable market access"
# # ================================================================
# #load social related data in geo dataset
# geo = merged_df.groupby('CustomerID').agg(
#     country_diversity = ('Country', 'nunique'),  # unique countries per customer
# ).reset_index()

# # PILLAR 2: Community Loyalty & Engagement
# # Source: Porter & Kramer (2011) - Shared Value Theory
# # "Repeat purchasing reflects stable, trust-based
# #  community engagement and economic inclusion"
# # ================================================================
# loyalty = merged_df.groupby('CustomerID').agg(
#     purchase_frequency = ('InvoiceNo',    'nunique'),  # repeat visits
#     active_days        = ('InvoiceDate',
#                           lambda x: (x.max() - x.min()).days + 1),
#     recency            = ('InvoiceDate',
#                           lambda x: (snapshot_date - x.max()).days),
# ).reset_index()

# # Engagement rate = how often they buy per active day
# loyalty['engagement_rate'] = (
#     loyalty['purchase_frequency'] /
#     loyalty['active_days'].replace(0, np.nan)
# ).fillna(0)


# # PILLAR 3: Product Access & Consumer Well-being
# # Source: UN SDG 12 - Responsible Consumption
# # "Access to diverse, sustainable products reflects
# #  consumer well-being and community product reach"
# # ================================================================
# access = merged_df.groupby('CustomerID').agg(
#     product_variety    = ('StockCode',    'nunique'),  # unique products accessed
#     total_quantity     = ('Quantity',     'sum'),      # volume of goods accessed
#     avg_spend          = ('TotalPrice',   'mean'),     # economic accessibility
# ).reset_index()

# #merge all 3 pillars            
# social = geo.merge(loyalty, on='CustomerID')
# social = social.merge(access, on='CustomerID')

# # ENTROPY WEIGHT METHOD
# # Source: Shannon (1948), Zeleny (1982)
# # OECD Composite Indicator Handbook (2008) Ch.6 —
# # "When no theoretical basis for weights exists,
# #  entropy weighting is the most objective method"
# # ================================================================
# X = social_norm[social_features].values + 1e-9  # avoid log(0)

# # Step 1: Normalise for entropy
# P = X / X.sum(axis=0)

# # Step 2: Entropy per feature
# k = 1 / np.log(len(P))
# entropy = -k * np.nansum(P * np.log(P), axis=0)

# # Step 3: Divergence
# divergence = 1 - entropy

# # Step 4: Final weights
# weights = divergence / divergence.sum()

# print("\n=== Entropy-derived Social Impact Weights ===")
# for feat, w in zip(social_features, weights):
#     print(f"  {feat:25s}: {w:.4f}")
# print(f"  {'Sum':25s}: {weights.sum():.4f}")

# #Final Social Impact Score:

# # Apply entropy weights
# social['social_impact'] = sum(
#     social_norm[feat] * w
#     for feat, w in zip(social_features, weights)
# )

# # Verify range
# print("\nSocial Impact Score stats:")
# print(social['social_impact'].describe())

# # ================================================================
# # INTERPRETATION:
# # High score = geographically diverse + loyal + high product access
# #              + eco-conscious purchasing behaviour
# # Low score  = single country + infrequent + narrow product range
# # ================================================================

# # Segment for reporting
# social['social_segment'] = pd.cut(
#     social['social_impact'],
#     bins=[0, 0.25, 0.50, 0.75, 1.0],
#     labels=['Low', 'Medium', 'High', 'Very High']
# )

# print("\nSocial segment distribution:")
# print(social['social_segment'].value_counts())

# SELV Formula with constraint α + β + γ = 1
# SELV = α×Financial Value + β×Social Impact + γ×Environmental Impact

# #SELV weights — tune based on your ESG policy alignment
# #value based on UN SDG retia
# alpha = 0.35   # Financial Value weight
# beta  = 0.35   # Social Impact weight
# gamma = 0.30   # Environmental Impact weight

# # Enforce constraint
# assert round(alpha + beta + gamma, 10) == 1.0, "Weights must sum to 1!"


# # Merge all 2 components
# master = rfm[['CustomerID', 'financial_value']].copy()
# master = master.merge(social[['CustomerID', 'social_impact']], on='CustomerID')
# master = master.merge(env[['CustomerID', 'environmental_impact','Carbon_Reduction_Ratio']],on='CustomerID')

# #carbon related data is already in main file


# #SELV = α×Financial Value + β×Social Impact + γ×Environmental Impact
# master['SELV'] = (
#     alpha * master['financial_value']      +
#     beta  * master['social_impact']        +
#     gamma * master['environmental_impact']
# )

# # Scale SELV to interpretable range (0-100)
# master['SELV_score'] = master['SELV'] * 100

# Add Churn Probability + XGBoost Predicted CLV

# # Merge RFM scores
# rfm['R_score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1]).astype(int)
# rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
# rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  5, labels=[1,2,3,4,5]).astype(int)
# rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# # merge rfm into master dataset
# master = master.merge(rfm[['CustomerID','Recency','Frequency',
#                             'Monetary','RFM_score']], on='CustomerID')
# # merge env into master dataset
# master = master.merge(env[['CustomerID','carbon_emission_kg']], on='CustomerID')

# # Churn label
# CHURN_DAYS = 90    # we take 90 days for tracking customer 
# master['is_churned'] = (master['Recency'] > CHURN_DAYS).astype(int)   # checking the customer last update more than 90 days, if condition is true then customer is churn


# features = ['Frequency','Monetary','RFM_score',
#             'financial_value','social_impact',
#             'environmental_impact','carbon_emission_kg','Carbon_Reduction_Ratio']



# X       = master[features]
# y_churn = master['is_churned']
# y_clv   = master['SELV_score']

# # train and test the s, y_clv, y_churn
# # test_size=0.2 due to records around thousands.
# # yc_tr ->  Churn target   , yc_te -> churn test 
# # ys_tr -> clv score 
# # X_train -> Features
# X_train, X_test, yc_tr, yc_te = train_test_split(X, y_churn, test_size=0.2, random_state=42)    
# _,       _,      ys_tr, ys_te = train_test_split(X, y_clv,   test_size=0.2, random_state=42)


# # Churn model
# churn_model = XGBClassifier(n_estimators=200, max_depth=4,
#                              learning_rate=0.05, eval_metric='logloss')        
# # n_estimators=200, means model build 200 trees sequences
# # max_depth -> complexity of each tree. value 4 is relatively shallow, 
# # 4 prevent from the complex , and random noise patterns.  
# # (The Step Size)learning_rate=0.05 : Lower learning rates (like 0.05) usually lead to better accuracy
# # eval_metric='logloss' (The Scoring System) : measure if the model is doing good /not.

# # training phase
# churn_model.fit(X_train, yc_tr)  #X_train is like question, yc_tr is like answer

# # predictions to Risk Assessment.
# # predict_proba-> Calculate score (0-> stay, 1-> churn)
# master['churn_prob'] = churn_model.predict_proba(X)[:,1]

# master['churn_prob']

# # SELV prediction model
# selv_model = XGBRegressor(
#     n_estimators=200, max_depth=3,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_alpha=0.1,
#     reg_lambda=1.0
# )                                                      # XGBRegressor predict -> how much customer are worth
# selv_model.fit(X_train, ys_tr)
# # for business strategy
# master['predicted_SELV'] = selv_model.predict(X)    # "Sustainability-Adjusted Value" to every single person 

# # PRIORITY MATRIX

# # CHURN PROB                  PREDICTED SELV                      STARTEGY 
# #   high                          high                            critical(high-valed customer about to leave, don't let them leave)
# #   low                           high                            loyal(are like stars. keep them happy)
# #   high                          low                             let go(high risk, no profitable)
# #   low                           low                             nurture(low risk, but try to upsell more sustainable products)



# # SELV Segmentation

# def selv_segment(row):
#     if row['SELV_score'] >= 75 and row['churn_prob'] < 0.3:
#         return 'Sustainable Champion'
#     elif row['SELV_score'] >= 60:
#         return 'High Value Green'
#     elif row['churn_prob'] > 0.7:
#         return 'At Risk'
#     elif row['environmental_impact'] > 0.7:
#         return 'Green Low Value'
#     elif row['financial_value'] > 0.7:
#         return 'High Value High Emitter'
#     else:
#         return 'Needs Nurturing'


# master['selv_segment'] = master.apply(selv_segment, axis=1)

# # Final output
# output_cols = ['CustomerID', 'Recency', 'Frequency', 'Monetary',
#                'RFM_score', 'financial_value', 'social_impact',
#                'environmental_impact', 'SELV_score', 'predicted_SELV',
#                'churn_prob', 'Carbon_Reduction_Ratio', 'selv_segment']

# output = master[output_cols].sort_values('SELV_score', ascending=False)
# output.to_csv('selv_clv_output.csv', index=False)

# output['churn_prob']

# print("\nSELV segment distribution:")
# output['selv_segment'].value_counts()

